# Animated scenario mapping with TimestampedGeoJson

Temporal maps can show sequences: facility openings, inspections, outbreaks, surveys, or extreme-weather impacts. This notebook builds a small time-enabled GeoJSON layer from facility points.

The lesson is about structure: a time property, a geometry, and a popup. Replace the sample rows with real operational data when available.

Video prompt for cognition: NASA data tools often emphasize time sliders and changing Earth systems. <iframe width="560" height="315" src="https://www.youtube.com/embed/DX0EGRaAf8I" title="My NASA Data Explorer" frameborder="0" allowfullscreen></iframe>

**Reflection questions:** What time interval is meaningful for the phenomenon? What events should not be animated because animation implies causality? How would missing dates be represented?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
fac = load_csv('health_facilities_training_points.csv').head(7)
features=[]
for i, r in fac.iterrows():
    features.append({
        'type':'Feature',
        'geometry': {'type':'Point', 'coordinates':[r.lon, r.lat]},
        'properties': {
            'time': f'2024-01-{1+i:02d}',
            'popup': r['name'],
            'icon': 'circle',
            'iconstyle': {'fillOpacity':0.8, 'stroke':'true', 'radius':8}
        }
    })
geo = {'type':'FeatureCollection', 'features':features}
m = folium.Map(location=[42.8,-75], zoom_start=5, tiles='CartoDB positron')
TimestampedGeoJson(geo, period='P1D', add_last_point=True, auto_play=False, loop=False).add_to(m)
add_standard_controls(m)
m